# Lab | Summarization evaluation using LangSmith
Let's revisit your capstone project 2? Well, sort of. Pick diffierent sets of data and re-run this notebook. Maybe parts of the dataset you used in your last project week. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications using LangSmith.

What did you learn? - Let's discuss that in class

## LangSmith - LangChain evaluation

> ⚠️ **Updated:** Original notebook loaded API keys from a `.env` file using `dotenv`. This version keeps that approach but loads `GROQ_API_KEY`, `HF_TOKEN`, and `LANGCHAIN_API_KEY` instead of the original OpenAI key.

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
HF_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')
LANGCHAIN_API_KEY = os.getenv('LANGCHAIN_API_KEY')

In [ ]:
# Listing all available models from GROQ
import requests
import json

url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(json.dumps(response.json(), indent=2))

In [3]:
#Importing Client from Langsmith
from langsmith import Client
client = Client(api_key=LANGCHAIN_API_KEY)

### Create Dataset


> ⚠️ **Updated:** Original used `ccdv/cnn_dailymail` with `version=` and `trust_remote_code=True` which are no longer supported. Updated to use `cnn_dailymail` with `name=` parameter.

In [4]:
from datasets import load_dataset
cnn_dataset = load_dataset("cnn_dailymail", name="3.0.0", token=HF_TOKEN)

d:\miniconda3\envs\langchain-eval\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def add_prefix(example):
    return {
        **example,
        "article": f"Summarize this news:\n{example['article']}"
    }

#cnn_dataset = cnn_dataset.map(add_prefix)

In [6]:
cnn_dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [7]:
cnn_dataset['train'][0]

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [8]:
#Get just a few news to test
MAX_NEWS=10
sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS)).map(add_prefix)

sample_cnn

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 10
})

The dataset contains three columns: article, highlights, and id. To use LangSmith, we need to create a dataset in LangSmith format.

LangSmith expects a prompt and a result. To achieve this, we will transform the article into a prompt by adding the prefix: "Summarize this news." As a result, we will use the content of highlights, which represents the summaries created by humans.

In [9]:
print(sample_cnn[0])

{'article': 'Summarize this news:\n(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC\'s founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians\' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki,

Now We have the Dataset with the prompt and the Reference Summary, it is time to create a Dataset in LangSmith with this information.
### Create the Dataset in Langsmith

The dataset in LangSmith is composed of an input, which is the prompt passed to the model for evaluation, and an output, which should contain what we expect the model to return.

In [10]:
import uuid
input_key = ['article']
output_key = ['highlights']

NAME_DATASET = "CNN_Summarization_Eval"

In [11]:
# Creates the dataset in LangSmith only if it doesn't already exist
if not client.has_dataset(dataset_name=NAME_DATASET):
    dataset = client.upload_dataframe(
        df=sample_cnn,
        input_keys=input_key,
        output_keys=output_key,
        name=NAME_DATASET,
        description="Test Embedding distance between model summarizations",
        data_type="kv"
    )
    print(f"✅ Dataset created: {NAME_DATASET}")
else:
    print(f"ℹ️ Dataset already exists, reusing: {NAME_DATASET}")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 285.23ba/s]


✅ Dataset created: CNN_Summarization_Eval


In this image, we can see an example from the dataset once it's been registered in LangSmith.

In the Input column, there is the prompt to be sent, while in the Output column, the expected output is stored.

When performing the comparison, the model will be given the prompt, and the Cosine distance between its response and the one stored in the sample dataset will be calculated.
<img src="./data/outputs/bart_summarization_input_output.png">

### Recovering Models From Hugging Face
Let's retrieve both models from HuggingFace. A base model and a model that has been fine-tuned using the training portion of this same dataset to generate summaries.

> ⚠️ **Updated:** Original used `HuggingFaceHub` from `langchain` which is deprecated, and `t5-base` / `flax-community/t5-base-cnn-dm` which are no longer accessible via the HuggingFace Inference API. Updated to use direct HTTP requests to the HuggingFace Router API with `facebook/bart-large-cnn` as the base model and `sshleifer/distilbart-cnn-12-6` as the fine-tuned alternative (both trained on CNN/DailyMail).

In [19]:
import sys
!{sys.executable} -m pip install sentence-transformers -q

import requests
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def hf_summarize(text, repo_id):
    API_URL = f'https://router.huggingface.co/hf-inference/models/{repo_id}'
    headers = {'Authorization': f'Bearer {HF_TOKEN}'}
    try:
        response = requests.post(API_URL, headers=headers, json={'inputs': text[:1024]})
        if response.status_code == 200:
            result = response.json()
            if isinstance(result, list):
                return result[0].get('summary_text') or result[0].get('generated_text', '')
            return str(result)
        return f'API Error: {response.status_code}'
    except Exception as e:
        return f'Error: {str(e)}'

# Base model: facebook/bart-large-cnn
def summarizer_base(inputs):
    return hf_summarize(inputs, 'facebook/bart-large-cnn')

# Fine-tuned model: sshleifer/distilbart-cnn-12-6 (replaces flax-community/t5-base-cnn-dm — 404)
def summarizer_finetuned(inputs):
    return hf_summarize(inputs, 'sshleifer/distilbart-cnn-12-6')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1239.48it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Defining Evaluator
The first step is to define an evaluator, where we specify the variables we want to evaluate. In our case, I have chosen to measure only the "embedding_distance."

I've left the "string_distance" as a comment in case you want to conduct a test with two evaluations instead of one.

> ⚠️ **Updated:** Original used `run_on_dataset` and `RunEvalConfig` from `langchain.smith` which have been removed in newer LangChain versions. Updated to use `langsmith.evaluate` with a custom `embedding_distance_evaluator` function that computes cosine distance using **sentence-transformers** (`all-MiniLM-L6-v2`) instead of OpenAI embeddings.

In [20]:
from langsmith import evaluate

def get_embedding(text):
    if not text or not str(text).strip():
        return np.zeros(384)  # all-MiniLM-L6-v2 outputs 384 dims
    return embedding_model.encode(str(text)[:8000])

def embedding_distance_evaluator(run, example):
    prediction = (run.outputs or {}).get('output', '')
    reference = (example.outputs or {}).get('highlights', '')
    emb1 = get_embedding(str(prediction))
    emb2 = get_embedding(str(reference))
    cosine_sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2) + 1e-9)
    return {'key': 'embedding_distance', 'score': 1 - cosine_sim}

#We are using just one of the multiple evaluators available on LangSmith.
#string_distance is left as a comment in case you want to add a second evaluator.
#string_distance_evaluator can be added to the evaluators list similarly.

### Running Evaluator
With the same configuration, we can launch two evaluations on the same dataset. One for each of the chosen models.

> ⚠️ **Updated:** Original used `run_on_dataset` with `summarizer_base` (t5-base). Updated to use `langsmith.evaluate` with `facebook/bart-large-cnn` as the base model since t5-base is no longer accessible via the HuggingFace Inference API.

In [14]:
def summarize_bart(inputs: dict):
    output = hf_summarize(inputs['article'], 'facebook/bart-large-cnn')
    return {'output': output}

bart_results = evaluate(
    summarize_bart,
    data=NAME_DATASET,
    evaluators=[embedding_distance_evaluator],
    experiment_prefix='BART-large-CNN',
    client=client,
)

View the evaluation results for experiment: 'BART-large-CNN-0fd1f575' at:
https://eu.smith.langchain.com/o/1f029fdc-0c26-4e75-94d7-431e0844adcd/datasets/4dd33c3f-8ac1-47f4-ba64-cac52ed74903/compare?selectedSessions=18cae958-d16a-407c-a9dd-479688ed3cf0




10it [00:49,  4.97s/it]


> ⚠️ **Updated:** `flax-community/t5-base-cnn-dm` returns a 404 — the model has been removed from the HuggingFace Hub. Replaced with `sshleifer/distilbart-cnn-12-6`, a distilled BART model fine-tuned on the same CNN/DailyMail dataset.

In [21]:
def summarize_distilbart(inputs: dict):
    output = hf_summarize(inputs['article'], 'sshleifer/distilbart-cnn-12-6')
    return {'output': output}

distilbart_results = evaluate(
    summarize_distilbart,
    data=NAME_DATASET,
    evaluators=[embedding_distance_evaluator],
    experiment_prefix='DistilBART-cnn-12-6',
    client=client,
)

View the evaluation results for experiment: 'DistilBART-cnn-12-6-88af5561' at:
https://eu.smith.langchain.com/o/1f029fdc-0c26-4e75-94d7-431e0844adcd/datasets/4dd33c3f-8ac1-47f4-ba64-cac52ed74903/compare?selectedSessions=2ede3dd6-1807-48c5-a8a6-7c2efc1d9369




10it [00:23,  2.38s/it]


In the image below you can see the comparison between the two HuggingFace models — **BART-large-CNN** and **DistilBART-cnn-12-6** — evaluated on the same dataset in LangSmith. Each row represents one article from the test set, with its embedding distance score.

<img src="./data/outputs/full_comparison_table_langsmith.png">

Well, since it has been so straightforward, why don't we try to make the comparison with an open source model? Let's use **Llama 3.1-8b** via the Groq API!

> ⚠️ **Updated:** Original used `from langchain.chat_models import ChatOpenAI` with `gpt-3.5-turbo`, which is deprecated and requires a paid API key. Updated to use the **Groq** client with `llama-3.1-8b-instant`, a free open source model.

In [16]:
import sys
!{sys.executable} -m pip install groq -q

from groq import Groq as GroqClient

groq_llm_client = GroqClient(api_key=GROQ_API_KEY)

def summarize_groq(inputs: dict):
    article = inputs['article'][:3000]
    response = groq_llm_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': article}],
        temperature=0.0
    )
    return {'output': response.choices[0].message.content or 'No summary generated'}

groq_results = evaluate(
    summarize_groq,
    data=NAME_DATASET,
    evaluators=[embedding_distance_evaluator],
    experiment_prefix='Groq-Llama3.1-8b',
    client=client,
)

View the evaluation results for experiment: 'Groq-Llama3.1-8b-6dd07e22' at:
https://eu.smith.langchain.com/o/1f029fdc-0c26-4e75-94d7-431e0844adcd/datasets/4dd33c3f-8ac1-47f4-ba64-cac52ed74903/compare?selectedSessions=22c706ae-6c46-4531-a1af-23e7499b3ad0




10it [00:09,  1.02it/s]


In the image below you can see the full comparison across all three models — **BART-large-CNN**, **DistilBART-cnn-12-6**, and **Groq Llama 3.1-8b** — side by side in LangSmith. Groq Llama 3.1-8b achieves the lowest average embedding distance, outperforming both fine-tuned HuggingFace models despite being a general-purpose open source model.

<img src="./data/outputs/cnn_summarization_models_overview.png">

The experiment with the Groq Llama 3.1-8b model has yielded the best results. And the best part — it's open source and runs through Groq's free inference API, so there's no cost involved.

Another crucial piece of information is that we can view performance data for the models. This data could also be useful for minimally evaluating our inference server.

---

## What We Built

We built a **multi-model summarization evaluation pipeline using LangSmith** to compare how well different LLMs summarize news articles against human-written references.

- **LangSmith dataset + evaluator**: 10 articles from CNN/DailyMail are uploaded as a reusable evaluation dataset with `article` as input and `highlights` as the reference output; a custom `embedding_distance_evaluator` measures semantic similarity between model output and reference using cosine distance on `all-MiniLM-L6-v2` embeddings.
- **Three model runners**: Separate evaluation functions wrap `facebook/bart-large-cnn` (HF Inference API), `sshleifer/distilbart-cnn-12-6` (HF Inference API), and `llama-3.1-8b-instant` via Groq, each logging results as named experiments in LangSmith.

This enables end-to-end automated evaluation of summarization quality across task-specific fine-tuned models and a general-purpose open source LLM, tracked and compared in a single LangSmith dashboard.

---

## Extensions We Implemented

### **API Health Check Cell**
Added a pre-flight diagnostic cell that tests both HuggingFace Inference API endpoints before running evaluations. This surfaces 404s and API errors immediately (as happened with `flax-community/t5-base-cnn-dm`) rather than discovering silent failures buried in LangSmith results after a full evaluation run.

### **Idempotent Dataset Creation**
Replaced the timestamped dataset name (`Summarize_dataset_<datetime>`) with a fixed name (`CNN_Summarization_Eval`) and added a `client.has_dataset()` guard so the dataset is only created on first run. Re-running the notebook reuses the existing dataset and avoids duplicate creation errors.

### **OpenAI-free Evaluation Stack**
Replaced OpenAI embeddings (required a paid API key) with local `sentence-transformers` (`all-MiniLM-L6-v2`) for the evaluator, and replaced `ChatOpenAI / gpt-3.5-turbo` with the Groq client running `llama-3.1-8b-instant`. The entire notebook now runs on free-tier APIs only (`GROQ_API_KEY`, `HF_TOKEN`, `LANGCHAIN_API_KEY`).

### **T5 Model Replacement**
Identified that `flax-community/t5-base-cnn-dm` was returning 404 (model removed from HuggingFace Hub), which produced misleadingly high embedding distance scores (~0.97) that looked like a result rather than a failure. Replaced it with `sshleifer/distilbart-cnn-12-6`, a verified working model trained on the same CNN/DailyMail dataset.

---

## What We Learned

| Topic | Key Insight |
|-------|-------------|
| **Embedding distance as a metric** | Cosine distance between model output and reference embeddings is a scalable, reference-based evaluation that requires no LLM judge — lower score means semantically closer to the human summary |
| **Silent API failures corrupt results** | A 404 from the HF Inference API returns a short error string, not an exception — without a health check this silently produces near-maximum distances (~0.97) that look like valid (bad) results |
| **Model compression trade-offs** | DistilBART is ~40% smaller than BART-large-CNN but scores worse (0.311 vs 0.279) with nearly double the standard deviation — compression hurts both quality and consistency |
| **General LLMs vs fine-tuned models** | Llama 3.1-8b (general-purpose, never trained on CNN/DailyMail) outperformed both fine-tuned HF models with avg distance 0.267 — modern instruction-tuned LLMs can match domain-specific fine-tuning |
| **Output length vs semantic similarity** | Llama produced ~986-char outputs vs ~315-char for BART/DistilBART; the reference highlights are short bullet points, yet Llama still won — verbosity alone doesn't determine embedding distance |
| **LangSmith experiment tracking** | Using `experiment_prefix` in `langsmith.evaluate` creates named, comparable runs; all experiments sharing the same dataset name can be visualised side-by-side in the LangSmith comparison view |
| **Idempotent notebook design** | Dataset creation and model availability checks should be guarded so a notebook can be re-run safely without accumulating duplicates or failing silently on stale external dependencies |